# Registro histórico: 04_mobilenet_onnx

Origem: `C00` — `Colab Notebooks/MobileNetTraining_02.ipynb`.

Tentativa de exportação de checkpoint MobileNet para ONNX.

**Limitações conhecidas:** Checkpoint ausente; configuração de exportação precisa ser validada. Nenhum ONNX foi incluído no material.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


In [ ]:
import os
import random
import shutil
from pathlib import Path

In [ ]:
!pip install --upgrade onnx onnxscript

In [ ]:
if not os.path.exists("/content/models"):
    print("Importando Modelo")
    from google.colab import drive
    drive.mount('/content/drive')

    # Copy dataset from Drive to Colab
    !cp -r "/content/drive/MyDrive/0-vgg/models" "/content/"
else:
    print("pasta de modelos já importada")

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

# Definir a arquitetura do modelo MobileNetV3
# Vamos usar o MobileNetV3 grande como base
model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)

# Modificar a camada classificadora para 2 classes (binário)
num_ftrs = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(num_ftrs, 2)

print("Arquitetura do modelo MobileNetV3 definida.")

In [ ]:
# Carregar os pesos do arquivo .pth
weights_path = "/content/models/MobileNetV3_SkinLesion_Binary_Classifier.pth"
model.load_state_dict(torch.load(weights_path, map_location=torch.device('cpu')))

print(f"Pesos carregados do arquivo: {weights_path}")

In [ ]:
import random
from pathlib import Path
from PIL import Image
import torchvision.transforms as transforms

# Diretório contendo as imagens de validação
val_dir = Path("/content/drive/MyDrive/0-vgg/base_dir/val_dir")

# Listar todas as subpastas (classes) no diretório de validação
classes = [d for d in val_dir.iterdir() if d.is_dir()]

# Selecionar uma classe aleatoriamente
random_class_dir = random.choice(classes)

# Listar todas as imagens dentro da classe selecionada
images = [f for f in random_class_dir.iterdir() if f.suffix in ['.jpg', '.jpeg', '.png']]

# Selecionar uma imagem aleatoriamente
random_image_path = random.choice(images)

print(f"Imagem selecionada aleatoriamente: {random_image_path}")

# Carregar a imagem
img = Image.open(random_image_path).convert('RGB')

# Definir as transformações (devem ser as mesmas usadas no treinamento)
# Adapte as transformações conforme necessário para o seu modelo
transform = transforms.Compose([
    transforms.Resize((224, 224)), # Redimensionar para o tamanho esperado pelo MobileNetV3
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Normalização ImageNet
])

# Aplicar as transformações e adicionar uma dimensão de batch
example_inputs = transform(img).unsqueeze(0)

print("Imagem carregada e transformada, pronta para ser usada como example_inputs.")

In [ ]:
import torch.nn.functional as F

onnx_program = torch.onnx.export(model, example_inputs,
                                 opset_version=9,
                                 dynamo=True)

In [ ]:
onnx_program.save("MobileNetV3Model_02.onnx")